# Install Packages

In [1]:
import re
import json
import os
from pathlib import Path
from datetime import datetime
from pprint import pprint

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# Import typing helpers for type hints 
from typing import TypedDict, Dict, List, Literal, Optional, Any

# BaseModel is the core class used to define structured data models 
from pydantic import BaseModel, Field, ConfigDict, model_validator, ValidationError

In [3]:
# Add project root to be part of search as I am running this notebook in a child folder
import sys
sys.path.append(os.path.abspath(".."))

# Import schemas
from schemas.content_blueprint import ContentBlueprint
from schemas.style_profile import StyleProfileStructure
from schemas.reflection_blueprint import ReflectionBlueprint

In [5]:
# Get project directory aka root folder
BASE_DIR = Path.cwd().parent
print(BASE_DIR)

/app


# Load API Key

In [6]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
api_key = os.getenv("OPENAI_API_KEY")

# Run LangGraph flow

I have a modified flow offline which will output artifacts to the root directory.  
This will collect the artifacts to run the evaluation below.

# Independent Evaluation Pipeline

The system includes a Reflection Agent used to give targeted feedback during generation-time revision. Separately, we have an independent evaluation pipeline outside the LangGraph loop. 

The project requirement is not only to evaluate the final result (via the Judging Agent) but also the process. We need to check if the steps, tools, etc were correct and efficient. In this process, the key questions circle around whether the output is in the correct format, and whether it is useful for the next step:

Style Extraction Agent
* Did the agent produce notes in the expected schema? (correctness)
* Did the notes actually reflect style characteristics? (usefulness)

Style Aggregation Agent
* Did it summarise the style chunk notes correctly? (usefulness)

Script Writing Agent
* Did it follow the outline format? (correctness)
* Did it follow the style profile given? (correctness)

Reflection Agent
* Precision: Of the items it flagged, how many were truly issues?

**Approach**

Style Extraction Agent
* Schema validity rate: `valid_outputs / total_outputs`
* Usefulness score: LLM judge to output an intermediate quality score (scale of 1-5)

Style Aggregation Agent
* Manually compile top styles per aspect, then see if the LLM managed to catch those top styles in their 6 options provided per style

Script Writing Agent
* Outline coverage score: Flatten the nested JSON content_outline, only include `must_include_points` and `must_include_facts`. `items covered / total items`.
* `Style_profile` fidelity: LLM judge to output an intermediate quality score (scale of 1-5)

Reflection Agent
* Compute precision: `true_issues_flagged` / `total_issues_flagged`

Output can be JSON with the score and reason
```json
{
  "score": 4,
  "reason": "Notes capture tone and rhetorical patterns clearly."
}
```

For the quality / usefulness scores on a scale of 1 to 5, rubrics and their definitions will be individually defined in the user_prompts

In [8]:
# Create a simple JSON schema (no need Pydantic)
schema = {
    "title": "evaluation",
    "type": "object",
    "properties": {
        "score": {"type": "integer"},
        "reason": {"type": "string"}
    },
    "required": ["score", "reason"]
}

## Load Artifacts for Evaluation

In [9]:
with open(BASE_DIR / "chunk_style_notes.json", "r") as f:
    chunk_style_notes_output = json.load(f)

In [10]:
with open(BASE_DIR / "style_profile.json", "r") as f:
    style_profile_output = json.load(f)

In [11]:
with open(BASE_DIR / "content_outline.json", "r") as f:
    content_outline_output = json.load(f)

In [12]:
with open(BASE_DIR / "style_feedback.json", "r") as f:
    style_feedback_output = json.load(f)

## Style Extraction Agent Process Evaluation

**Step 1: Schema Validity**  
Validate against pydantic schema

In [13]:
# Compute schema validity
total_outputs = len(chunk_style_notes_output)

valid_outputs = 0
invalid_outputs = 0
invalid_output_indices = []

for i, note in enumerate(chunk_style_notes_output):
    try:
        StyleProfileStructure.model_validate_json(json.dumps(note)) # convert back from Python dict to JSON
        valid_outputs += 1
    except ValidationError:
        invalid_outputs += 1
        invalid_output_indices.append(i)

schema_validity_rate = valid_outputs / total_outputs if total_outputs > 0 else 0.0

print("valid_outputs:", valid_outputs)
print("invalid_outputs:", invalid_outputs)
print("invalid_indices:", invalid_output_indices)
print("schema_validity_rate:", schema_validity_rate)

valid_outputs: 14
invalid_outputs: 0
invalid_indices: []
schema_validity_rate: 1.0


**Step 2: Usefulness of Style Notes per Chunk**  
Use LLM as a Judge to score how well the style notes match up to the chunk of text

In [14]:
style_ext_eval_sys_pmpt = """
You are an impartial evaluation model. Follow the instructions strictly.
"""

In [15]:
def build_style_ext_eval_user_pmpt(chunk, chunk_style_notes):
    return f"""
Evaluate whether the style notes accurately describe the speaking style in the speech excerpt.

Check whether the stylistic observations in the style notes can be logically inferred from the excerpt. Focus only on *how* the speaker communicates, not the topic.

Rules:
- Only give high scores if the notes are clearly supported by specific evidence in the excerpt.
- If a claim in the notes is not visible in the excerpt, it should lower the score.
- When uncertain, prefer the lower score.
- Explanation must be concise (maximum 5 sentences).

Scoring:
1 = Not supported by the excerpt
3 = Partially supported or vague
5 = Clearly supported and well-reflected
Use 2 or 4 if the match is between levels.

Speech excerpt:
\"\"\"
{chunk}
\"\"\"

Style notes:
{chunk_style_notes}

Return:
- score (1–5)
- reason (brief explanation citing evidence or missing support)
"""

In [18]:
# Compute usefulness score
# output a usefulness score per comparison
# collect all the scores and average them

# Extract chunks .txt files and sort in order to match with chunk_style_notes
chunks_folder = BASE_DIR / "data" / "samples_chunks"
files = sorted(
    chunks_folder.glob("*.txt"),
    key=lambda x: int(x.stem.split("_")[-1])  # extract number
)
chunks_texts = []
for file in files:
    with open(file, "r", encoding="utf-8") as f:
        chunks_texts.append(f.read())

# init LLM
style_ext_eval_llm = ChatOpenAI(
    model="gpt-4.1-mini", 
    temperature=0
).with_structured_output(schema)

usefulness_scores = []
# Loop through each chunk, chunk notes to get a score
for c, cn in zip(chunks_texts, chunk_style_notes_output):
    score_obj = style_ext_eval_llm.invoke([
            SystemMessage(content=style_ext_eval_sys_pmpt),
            HumanMessage(
                content=build_style_ext_eval_user_pmpt(
                    chunk=c, 
                    chunk_style_notes=cn
                )
            )
        ])
    print(score_obj)
    usefulness_scores.append(score_obj)

{'score': 5, 'reason': "The excerpt clearly shows a confrontational and direct tone with candid and challenging language, such as 'you are living in a dream world' and 'please be at least in a position to comprehend.' The vocabulary is plainspoken and informal, with colloquial expressions like 'leader man.' Syntax includes short declarative sentences and direct audience address ('Friends and fellow-citizens,' 'If you read...'). Rhetorical devices like direct address and contrast framing are evident. The argument structure involves blunt issue naming and challenges to credibility, matching the style notes well."}
{'score': 5, 'reason': "The style notes are clearly supported by the excerpt: the tone is conversational and candid with informal pronouns and colloquial expressions ('you know', 'smack you'). The speaker uses short declarative sentences and parenthetical insertions ('I literally say ‘everybody’'). Rhetorical devices like direct audience address and anaphora ('you know') are pr

In [19]:
# Compute average usefulness score
avg_score = sum(item['score'] for item in usefulness_scores) / len(usefulness_scores)
print(avg_score)

5.0


In [20]:
# Manually check against the text and see if the LLM's analysis makes sense
for c, t in zip(chunks_texts, chunk_style_notes_output):
    print("CHUNK:", c)
    print("\n\nSTYLE NOTES:", t)

CHUNK: Friends and fellow-citizens,

If you read, and your understand only the English Language, then you are at a very grave disadvantage because you really don’t know what is going on in a large part of Singapore . If you believe that the Straits Times and the New Nation is what Singapore is about, then you are living in a dream world.

Before you even pretend to want to be a candidate, let alone a leader man, or as Mr. Jeyaretnam hopes to be leader of Her Majesty’s Opposition, says he over Radio Singapore -- I say, please be at least in a position to comprehend what is taking place.


STYLE NOTES: {'tone': ['confrontational', 'direct', 'candid', 'challenging', 'informal'], 'avoid': ['motivational slogans', 'corporate buzzwords', 'sentimental phrasing', 'polished rhetoric', 'abstract language'], 'lexical_preferences': ['plainspoken vocabulary', 'colloquial expressions', 'concrete nouns', 'informal address terms', 'repeated key terms'], 'syntax': ['short declarative sentences', 'direc

## Style Aggregation Agent Process Evaluation

Manually compile the style attributes and take the top 6 in each aspect. Check if the LLM had included any of these top 6 inside the style_profile.

In [21]:
# Tally top style attributes from chunk_style_notes
# This is what a human would do manually

from collections import defaultdict, Counter
tally = defaultdict(Counter)

for note in chunk_style_notes_output:
    for aspect, phrases in note.items():
        tally[aspect].update(p.lower() for p in phrases)

for aspect, counter in tally.items():
    print(f"\n{aspect}")
    for attr, count in counter.most_common(6):
        print(f"  {attr}: {count}")


tone
  direct: 7
  pragmatic: 7
  informal: 6
  conversational: 5
  authoritative: 5
  slightly confrontational: 4

avoid
  motivational slogans: 14
  corporate buzzwords: 14
  sentimental phrasing: 14
  overly formal language: 6
  abstract language: 4
  overly formal diction: 3

lexical_preferences
  colloquial expressions: 13
  plainspoken vocabulary: 8
  concrete nouns: 8
  repeated key terms: 7
  numerical comparisons: 2
  specific numerical data: 2

syntax
  parenthetical insertions: 12
  short declarative sentences: 7
  parallel constructions: 6
  abrupt pivots: 4
  rhetorical questions: 4
  long complex sentences: 4

rhetorical_devices
  contrast framing: 14
  direct audience address: 11
  rhetorical questions: 9
  anecdotal evidence: 6
  anaphora: 4
  repetition for emphasis: 3

argument_structure
  blunt issue naming: 14
  example-driven reasoning: 11
  anticipation of objections: 9
  problem-consequence framing: 9
  escalation through concrete examples: 5
  challenge to cred

In [22]:
style_profile_output

{'tone': ['direct',
  'authoritative',
  'conversational',
  'informal',
  'slightly confrontational',
  'pragmatic'],
 'avoid': ['motivational slogans',
  'corporate buzzwords',
  'sentimental phrasing',
  'abstract language',
  'overly formal language',
  'poetic language'],
 'lexical_preferences': ['plainspoken vocabulary',
  'colloquial expressions',
  'concrete nouns',
  'repeated key terms',
  'specific proper nouns',
  'numerical data'],
 'syntax': ['short declarative sentences',
  'parenthetical insertions',
  'sentence fragments',
  'parallel constructions',
  'abrupt pivots',
  'rhetorical questions'],
 'rhetorical_devices': ['direct audience address',
  'contrast framing',
  'anaphora',
  'rhetorical questions',
  'anecdotal evidence',
  'repetition for emphasis'],
 'argument_structure': ['blunt issue naming',
  'example-driven reasoning',
  'problem-consequence framing',
  'anticipation of objections',
  'escalation through concrete examples'],
 'audience_relationship': ['d

In [25]:
# Check if the LLM's attributes mentioned per aspect are inside the top 6 styles

# Store top N aspects in a new variable
top_attrs = {}
for aspect, counter in tally.items():
    top_attrs[aspect] = [attr for attr, count in counter.most_common(6)]

# Tally how many LLM's attributes were part of the top 6 styles
style_agg_results = {}
for aspect, attrs in style_profile_output.items():
    in_top_attrs = sum(attr in top_attrs[aspect] for attr in attrs)
    style_agg_results[aspect] = f"{in_top_attrs}/{len(attrs)}"

pprint(style_agg_results)

{'argument_structure': '5/5',
 'audience_relationship': '4/6',
 'avoid': '5/6',
 'lexical_preferences': '4/6',
 'rhetorical_devices': '6/6',
 'syntax': '5/6',
 'tone': '6/6'}


Can be seen that LLM hallucinated some of the attributes in the `style_profile`

## Script Writing Agent Process Evaluation

**Section 1: Check Coverage Percentage**  
Use LLM to identify the no. of points and facts from the outline that were covered  
Then programmatically compute the coverage percent.

In [26]:
# Import schemas
from schemas.script_eval import ScriptEvaluation

In [27]:
# Export script drafts
speech_drafts_folder = BASE_DIR / "data" / "speech_drafts"
files = sorted(speech_drafts_folder.glob("*.txt"))
draft_texts = []
for file in files:
    with open(file, "r", encoding="utf-8") as f:
        draft_texts.append(f.read())

In [28]:
# Filter the content_outline JSON to only include purpose, must_include_points and must_include_facts
filtered_outline = [
    {
        "section_id": section["id"],
        "section_purpose": section["purpose"],
        "must_include_points": section.get("must_include_points", []),
        "must_include_facts": section.get("must_include_facts", [])
    }
    for section in content_outline_output.get("ted_sections", [])
]

pprint(filtered_outline)

[{'must_include_facts': ['In 2025, AI models from Google DeepMind and OpenAI '
                         'achieved gold medal-level scores at the '
                         'International Mathematical Olympiad.',
                         "GovTech's 'Transcribe' AI is used in Singapore's "
                         'civil service call centres to transcribe '
                         'conversations in all four official languages plus '
                         'Singlish, generating summaries so officers can focus '
                         'on serving the public.',
                         'AI helps Singaporeans plan travel by recommending '
                         'destinations and generating personalised '
                         'itineraries.'],
  'must_include_points': ['AI has transformed from struggling with basic tasks '
                          'to achieving advanced capabilities within a few '
                          'years.',
                          'AI is already widely u

In [30]:
# Convert ground truth outline (filtered_outline) to tally also
outline_tally = []
for section in filtered_outline:
    outline_tally.append({
            "id": section["section_id"],
            "num_matched_facts": len(section["must_include_facts"]),
            "num_matched_points": len(section["must_include_points"])
    })

pprint(outline_tally)

[{'id': 'TS1', 'num_matched_facts': 3, 'num_matched_points': 2},
 {'id': 'TS2', 'num_matched_facts': 3, 'num_matched_points': 2},
 {'id': 'TS3', 'num_matched_facts': 3, 'num_matched_points': 1},
 {'id': 'TS4', 'num_matched_facts': 3, 'num_matched_points': 3},
 {'id': 'TS5', 'num_matched_facts': 1, 'num_matched_points': 3}]


In [31]:
script_eval_sys_pmpt = """
You are an impartial evaluation model. Follow the instructions strictly.
"""

In [32]:
def build_script_eval_user_pmpt(filtered_outline, speech_draft):
    return f"""
You are evaluating whether a speech draft covers the required content in a content outline.

Task:
For each section in the content outline:
1. Identify the part(s) of the speech draft that best match the section_purpose.
2. Check which must_include_facts and must_include_points are present in those parts.

Instructions:
- Use the content_outline structure as the template for your JSON output.
- For each section, include only the facts and points that are found in the speech draft.
- If a fact or point is not found, omit it entirely from the JSON.
- Treat paraphrased or equivalent meaning as a valid match.
- When uncertain, do not include the item.
- Use the exact original strings from content_outline (no rewriting or summarising).
- Do not force a 1:1 mapping between sections and paragraphs.

Content outline:
{json.dumps(filtered_outline, ensure_ascii=False, indent=2)}

Speech draft:
\"\"\"
{speech_draft}
\"\"\"
"""

In [33]:
# Init LLM to compute items covered in the script / total items.
script_eval_llm = ChatOpenAI(
    model="gpt-4.1-mini", 
    temperature=0
).with_structured_output(ScriptEvaluation)

coverage_objects = []
# Loop through each draft, compare with the filtered_content outline to check the points that were covered
for d in draft_texts:
    coverage_obj = script_eval_llm.invoke([
            SystemMessage(content=script_eval_sys_pmpt),
            HumanMessage(
                content=build_script_eval_user_pmpt(
                    filtered_outline=filtered_outline, 
                    speech_draft=d
                )
            )
        ])
    dumped = coverage_obj.model_dump(mode="json")
    coverage_objects.append(dumped)

In [34]:
# See a sample of the output per draft
pprint(coverage_objects[0]['eval_sections'])

[{'id': 'TS1',
  'matched_facts': ['In 2025, AI models from Google DeepMind and OpenAI '
                    'achieved gold medal-level scores at the International '
                    'Mathematical Olympiad.',
                    "GovTech's 'Transcribe' AI is used in Singapore's civil "
                    'service call centres to transcribe conversations in all '
                    'four official languages plus Singlish, generating '
                    'summaries so officers can focus on serving the public.',
                    'AI helps Singaporeans plan travel by recommending '
                    'destinations and generating personalised itineraries.'],
  'matched_points': ['AI has transformed from struggling with basic tasks to '
                     'achieving advanced capabilities within a few years.',
                     'AI is already widely used by Singaporeans in practical '
                     'ways, such as travel planning and government services.']},
 {'id': 'TS2',

In [35]:
# Compute the no. of matched_points and matched_facts in each draft
coverage_tallies = []
for obj in coverage_objects:
    tally = []
    for section in obj["eval_sections"]:
        tally.append({
            "id": section["id"],
            "num_matched_facts": len(section["matched_facts"]),
            "num_matched_points": len(section["matched_points"])
        })
    coverage_tallies.append(tally)

In [36]:
# See how the output looks like
coverage_tallies

[[{'id': 'TS1', 'num_matched_facts': 3, 'num_matched_points': 2},
  {'id': 'TS2', 'num_matched_facts': 3, 'num_matched_points': 2},
  {'id': 'TS3', 'num_matched_facts': 3, 'num_matched_points': 1},
  {'id': 'TS4', 'num_matched_facts': 3, 'num_matched_points': 3},
  {'id': 'TS5', 'num_matched_facts': 1, 'num_matched_points': 3}],
 [{'id': 'TS1', 'num_matched_facts': 3, 'num_matched_points': 2},
  {'id': 'TS2', 'num_matched_facts': 3, 'num_matched_points': 2},
  {'id': 'TS3', 'num_matched_facts': 3, 'num_matched_points': 1},
  {'id': 'TS4', 'num_matched_facts': 3, 'num_matched_points': 3},
  {'id': 'TS5', 'num_matched_facts': 1, 'num_matched_points': 3}],
 [{'id': 'TS1', 'num_matched_facts': 3, 'num_matched_points': 2},
  {'id': 'TS2', 'num_matched_facts': 3, 'num_matched_points': 2},
  {'id': 'TS3', 'num_matched_facts': 3, 'num_matched_points': 1},
  {'id': 'TS4', 'num_matched_facts': 3, 'num_matched_points': 3},
  {'id': 'TS5', 'num_matched_facts': 1, 'num_matched_points': 3}]]

In [37]:
# Compute overall outline tally per draft
total_in_outline = sum(s['num_matched_facts'] for s in outline_tally) + sum(s['num_matched_points'] for s in outline_tally)

for draft_tally in coverage_tallies:
    items_covered = sum(s['num_matched_facts'] for s in draft_tally) + sum(s['num_matched_points'] for s in draft_tally)
    print(f"coverage percent: {items_covered}/{total_in_outline}")

coverage percent: 24/24
coverage percent: 24/24
coverage percent: 24/24


The script contains all the required points and facts

**Section 2: Check `style_profile` fidelity**  
Use LLM as a judge approach, judge on a scale of 1 to 5

In [38]:
voice_eval_sys_pmpt = """
You are an impartial evaluation model. Follow the instructions strictly.
"""

In [39]:
def build_voice_eval_user_pmpt(style_profile, speech_draft):
    return f"""
You are evaluating STYLE MATCH only.

Task:
Assess how closely the speech draft matches the speaking style described in the Style Profile.

Instructions:
- Evaluate only stylistic alignment, not whether the speech is good, accurate, persuasive, or well-structured unless those qualities are explicitly part of the style profile.
- Base your judgment on observable signals in the draft.
- Compare the draft against the profile traits. Do not infer style traits that are not stated or strongly implied.
- If the profile includes multiple traits, consider the overall balance, not just one matching trait.

Scoring rubric:
1 = Very poor match; the described style is mostly absent or contradicted
2 = Weak match; a few traits appear, but most are missing
3 = Partial match; some important traits are evident, others are missing or inconsistent
4 = Strong match; most important traits are present with minor gaps
5 = Excellent match; the style is clearly and consistently reflected

Speech draft:
\"\"\"
{speech_draft}
\"\"\"

Style Profile:
\"\"\"
{style_profile}
\"\"\"
"""

In [40]:
# Init LLM 
voice_eval_llm = ChatOpenAI(
    model="gpt-4.1-mini", 
    temperature=0
).with_structured_output(schema)

fidelity_scores = []
# Loop through each draft, compare with style_profile to check if the user's voice is present
for d in draft_texts:
    score_obj = voice_eval_llm.invoke([
            SystemMessage(content=voice_eval_sys_pmpt),
            HumanMessage(
                content=build_voice_eval_user_pmpt(
                    style_profile=style_profile_output, 
                    speech_draft=d
                )
            )
        ])
    fidelity_scores.append(score_obj)

In [41]:
pprint(fidelity_scores)

[{'reason': 'The speech draft excellently reflects the style profile. It uses '
            'a direct, authoritative, yet conversational and informal tone, '
            "including colloquial expressions like 'Just look around' and 'And "
            "if you’ve ever planned a trip recently.' The vocabulary is "
            'plainspoken with concrete nouns and specific proper nouns (e.g., '
            'GovTech, Tuas Port, SkillsFuture). Numerical data is frequently '
            'cited, supporting the pragmatic style. Syntax features short '
            'declarative sentences, parenthetical insertions, and abrupt '
            "pivots (e.g., 'That’s not just impressive; it’s a game "
            "changer.'). Rhetorical devices such as direct audience address "
            "('Good evening, everyone.'), contrast framing, anaphora, "
            'rhetorical questions, and anecdotal evidence are present. The '
            'argument structure bluntly names issues, uses example-driven '
    

In [42]:
# Compute average fidelity score
fidelity_avg_score = sum(item['score'] for item in fidelity_scores) / len(fidelity_scores)
print(fidelity_avg_score)

5.0


## Reflection Agent Process Evaluation

Precision: Of the items it flagged, how many were truly issues?  
Check the latest evaluation as an indication of the quality of this agent.

In [43]:
ref_eval_sys_pmpt = """
You are an impartial evaluation model. Follow the instructions strictly.
"""

In [44]:
def build_ref_eval_user_pmpt(style_feedback):
    return f"""
You are evaluating the precision of a Reflection Agent.

The Reflection Agent was explicitly instructed to raise ISSUES ONLY.
Your job is to judge whether the items in its output are actually issues.

Definitions:
- A true issue is a statement that identifies a genuine problem, omission, weakness, ambiguity, unsupported claim, unmet requirement, or deficiency.
- A non-issue is a statement that says something is fine, adequate, present, clear, within budget, met, or otherwise not problematic.
- Neutral observations, praise, confirmations, and status updates are also non-issues.
- An ambiguous item is unclear or borderline. Treat ambiguous items as NOT true issues when computing precision.

What counts as an evaluative item:
Any output statement that assesses a part of the draft, including top-level fields and section-level fields such as purpose, narrative_role, transition_out, word_budget, and issue descriptions.

Task:
1. Read the Reflection Agent output.
2. Count every evaluative item it produced.
3. Classify each as one of:
   - true_issue
   - non_issue
   - ambiguous
4. Compute:
   - total_flagged_items
   - true_issues
   - non_issues
   - ambiguous_items
   - precision = true_issues / total_flagged_items
5. Be strict. If an item says something is adequately covered, present, clear, met, or within budget, treat it as a non-issue.

Return ONLY in this format:

total_flagged_items: <number>
true_issues: <number>
non_issues: <number>
ambiguous_items: <number>
precision: <number between 0 and 1>
reason: <brief explanation in 2-4 sentences>

Additionally, list ALL items classified as non_issue.
non_issue_items:
- <item>
- <item>
...

Reflection Agent output:
{style_feedback}
"""

In [45]:
# Init LLM 
ref_eval_llm = ChatOpenAI(
    model="gpt-4.1-mini", 
    temperature=0
)

# Break down the feedback and let LLM evaluate part by part so it is not overwhelmed
content_issues = style_feedback_output["content_issues"]
style_issues = style_feedback_output["style_issues"]

# Review content_issues
content_eval_result = ref_eval_llm.invoke([
    SystemMessage(content=ref_eval_sys_pmpt),
    HumanMessage(content=build_ref_eval_user_pmpt(style_feedback=content_issues))
])
# Review style_issues
style_eval_result = ref_eval_llm.invoke([
    SystemMessage(content=ref_eval_sys_pmpt),
    HumanMessage(content=build_ref_eval_user_pmpt(style_feedback=style_issues))
])

The results vary, it's not very deterministic, but doesnt vary too much.  
Can be considered a rough gauge when I manually checked, coz if human check it is too tedious.

In [46]:
print(content_eval_result.content)

total_flagged_items: 4  
true_issues: 1  
non_issues: 2  
ambiguous_items: 1  
precision: 0.25  
reason: The Reflection Agent raised one clear issue about the lack of specific examples to illustrate AI's impact, which is a genuine deficiency. Two items stating that points are "covered adequately" are non-issues since they confirm adequacy rather than problems. The observation about the hook is ambiguous because it is unclear if it identifies a problem or just describes content. Thus, only one true issue out of four items yields a precision of 0.25.  

non_issue_items:  
- "AI has transformed from struggling with basic tasks to achieving advanced capabilities within a few years." (Covered adequately.)  
- "AI is already widely used by Singaporeans in practical ways, such as travel planning and government services." (Covered adequately.)


In [53]:
pprint(style_feedback_output["content_issues"])

{'big_idea': "By embracing AI thoughtfully and prioritizing Singaporeans' "
             'skills and welfare, Singapore can stay competitive and create '
             'good jobs in a dynamic future economy.',
 'hook': {'description': 'Open by highlighting how AI has rapidly evolved and '
                         'is already integrated into everyday life in '
                         'Singapore, affecting travel, government services, '
                         'and more.',
          'type': 'observation'},
 'ted_sections': [{'id': 'TS1',
                   'must_include_facts': [{'issue': 'Fact is mentioned but '
                                                    'lacks specific examples '
                                                    'or detail to fully '
                                                    'illustrate impact.',
                                           'item': 'AI helps Singaporeans plan '
                                                   'travel by recommendi

In [52]:
print(style_eval_result.content)

total_flagged_items: 8
true_issues: 7
non_issues: 1
ambiguous_items: 0
precision: 0.875
reason: The Reflection Agent mostly raised genuine issues about framing, tone, lexical choices, and rhetorical devices that could be improved. One item explicitly states "no issues noted," which is a non-issue since it confirms adequacy rather than identifying a problem. All other items identify deficiencies or weaknesses, so they are true issues.

non_issue_items:
- Some sentence fragments are used effectively; no issues noted.


In [50]:
pprint(style_feedback_output["style_issues"])

{'argument_structure': [{'issue': 'Job displacement concerns are acknowledged '
                                  'but could be framed more bluntly to '
                                  'emphasize urgency.',
                         'item': 'problem-consequence framing'}],
 'audience_relationship': [{'issue': 'The script is mostly inclusive and '
                                     'conversational but lacks moments that '
                                     'directly challenge opposing views.',
                            'item': 'challenging opponents'}],
 'avoid': [{'issue': 'The closing contains phrases that verge on slogans, '
                     "e.g., 'solid ideas that deliver results; innovation "
                     "grounded in reality; and unity that binds us together'.",
            'item': 'motivational slogans'},
           {'issue': "Phrases like 'better lives for all Singaporeans' and "
                     "'side by side' introduce sentimental tone contrary to "
  